# Stack Overflow Annual Developer Survey 2025 — Analytical Deep Dive

**Project Name:** Poll Results Visualizer  
**Project Author:** Akash Das  
**Milestone:** Phase 5 of 8 — Analysis (Methodological Review & Statistical Corrections)  
**Primary Dataset Input:** `data/processed/survey_results_cleaned.parquet` (49,191 rows × 176 columns)  
**Methodological Standard:** Strict non-causal observational inference, multivariable OLS with heteroskedasticity-robust (HC1) standard errors, multiple testing adjustments (Benjamini–Hochberg FDR and Bonferroni), contingency tables with exact survival functions, and ANOVA effect sizes ($\eta^2$, $\omega^2$).

---

## Executive Summary & Research Scope

Following the Phase 5 methodological review, this notebook executes a fully audited, reproducible statistical and econometric investigation into four central research questions:

1. **Research Question 1 (RQ1) — AI Trust Dynamics and Respondent Characteristics:**  
   Examines reported developer trust in the accuracy of AI developer tools (`AIAcc`), auditing the exact sample flow, career seniority brackets (`WorkExp_num`), and programming language ecosystems, with multiple hypothesis testing corrections across the family of language tests.
2. **Research Question 2 (RQ2) — Technology Stack and Conditional Compensation Associations:**  
   Estimates conditional compensation associations of specialized technologies (Rust, Go, Python, TypeScript, AWS, Docker) using a multivariable Mincerian OLS framework controlling for geography, experience, and role. Evaluates heteroskedasticity via Breusch-Pagan diagnostics, compares conventional vs. White HC1 robust standard errors, reports Variance Inflation Factors (VIF), and applies false-discovery-rate control.
3. **Research Question 3 (RQ3) — Autonomous AI Agent Adoption and Workplace Outcomes:**  
   Investigates the 6-tier taxonomy of autonomous AI agent adoption (`AIAgents`) against AI sentiment (`AISent`), code friction (`AIFrustration`), and developer job satisfaction (`JobSat`). Evaluates ANOVA effect sizes ($\eta^2$, $\omega^2$), post-hoc pairwise comparisons with Bonferroni adjustment, and audits the $2 \times 2$ sentiment contingency table.
4. **Research Question 4 (RQ4) — Remote Work Dynamics, Selection Missingness, and Compensation:**  
   Audits the non-response mechanism in `RemoteWork`, establishes that missingness is not MCAR with respect to `MainBranch`, and estimates conditional compensation differences across working arrangements with sensitivity analyses for complete-case selection.


## Section 1: Environment Setup & Analytical Sample Flow

We import standard numerical libraries alongside our pure-Python statistical engine (`src.analysis`).


In [ ]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path(os.path.abspath(".")).resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import math
import json
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px

# Mock Plotly show for headless verification
go.Figure.show = lambda self, *args, **kwargs: None

from src.analysis import (
    normal_cdf,
    normal_p_value,
    chi2_p_value,
    f_distribution_p_value,
    calculate_contingency_and_chi2,
    calculate_odds_ratios,
    fit_ols,
    calculate_vif,
    compare_groups_continuous,
    adjust_p_values,
)

print(f"Python environment: {sys.version.split()[0]}")
print(f"Working directory: {PROJECT_ROOT}")


In [ ]:
# Load canonical processed dataset
PARQUET_PATH = PROJECT_ROOT / "data" / "processed" / "survey_results_cleaned.parquet"
df = pd.read_parquet(PARQUET_PATH)

print(f"Canonical Dataset Loaded:")
print(f"  Total Respondents: {df.shape[0]:,}")
print(f"  Total Attributes:  {df.shape[1]:,}")


### 1.1 Reconciled Sample Flow Accounting
To resolve historical count discrepancies, we formally document the complete sample flow across the survey instruments:

- **Total Survey Respondents:** $N = 49,191$
- **`AISelect` Answered:** $N = 33,720$ ($68.55\%$), Missing: $N = 15,471$ ($31.45\%$)
- **Active AI Tool Users:** $N = 26,469$ ($53.81\%$), defined as respondents selecting Daily ($15,883$), Weekly ($5,958$), or Monthly/Infrequently ($4,628$) in `AISelect`.
- **`AIAcc` Answered:** $N = 33,297$ ($67.69\%$), Missing: $N = 15,894$ ($32.31\%$)
- **Active AI Users Answering `AIAcc`:** Exactly **$N = 26,126$** ($53.11\%$ of total respondents, $78.46\%$ of the `AIAcc` answered cohort).
- **Non-Active AI Users Answering `AIAcc`:** $N = 7,171$ ($5,380$ non-planners, $1,754$ planners, $37$ missing `AISelect`).
- *Note on Preliminary Reporting:* An earlier preliminary run reported $19,733$ active users based on an ad-hoc filter on `AIAgents` + `LearnCodeAI`. The authoritative survey instrument establishing active AI tool usage is `AISelect` ($N = 26,469$), of whom $26,126$ answered `AIAcc`.


In [ ]:
# Verify Sample Flow
n_total = len(df)
n_aiselect_ans = int(df["AISelect"].notna().sum())
n_aiacc_ans = int(df["AIAcc"].notna().sum())

active_labels = [
    "Yes, I use AI tools daily",
    "Yes, I use AI tools weekly",
    "Yes, I use AI tools monthly or infrequently",
]
active_ai_mask = df["AISelect"].isin(active_labels)
n_active_ai = int(active_ai_mask.sum())
n_active_ai_in_acc = int((active_ai_mask & df["AIAcc"].notna()).sum())
n_nonactive_in_acc = int((~active_ai_mask & df["AIAcc"].notna()).sum())

flow_df = pd.DataFrame([
    {"Stage / Filter": "1. Total survey respondents", "Count": n_total, "% Total": 100.0},
    {"Stage / Filter": "2. AISelect answered", "Count": n_aiselect_ans, "% Total": round(n_aiselect_ans / n_total * 100, 2)},
    {"Stage / Filter": "3. Active AI users (Daily + Weekly + Monthly in AISelect)", "Count": n_active_ai, "% Total": round(n_active_ai / n_total * 100, 2)},
    {"Stage / Filter": "4. AIAcc answered (Overall analytical denominator)", "Count": n_aiacc_ans, "% Total": round(n_aiacc_ans / n_total * 100, 2)},
    {"Stage / Filter": "5. Active AI users answering AIAcc (Subpopulation)", "Count": n_active_ai_in_acc, "% Total": round(n_active_ai_in_acc / n_total * 100, 2)},
    {"Stage / Filter": "6. Non-active AI users answering AIAcc", "Count": n_nonactive_in_acc, "% Total": round(n_nonactive_in_acc / n_total * 100, 2)},
])
print("Sample Flow Accounting Table:")
print(flow_df.to_string(index=False))


## Section 2: Research Question 1 (RQ1) — AI Trust Dynamics

### 2.1 Audit of Seniority Brackets & Crude Odds Ratios
We examine developer trust in AI accuracy (`AIAcc`) across career experience brackets (`WorkExp_num`) among $N = 30,780$ complete cases.
- **Outcome:** Distrust (Somewhat distrust + Highly distrust) vs. Not Distrust (Trust + Neutral).
- **Reference Group:** Entry-level developers (<2 years experience).
- **Odds Ratio Formula:** For bracket $j$ with cell counts $a$ (distrust) and $b$ (not distrust) vs. baseline $c$ (distrust) and $d$ (not distrust):
  $$\text{OR} = \frac{a \cdot d}{b \cdot c}$$
  Standard error of $\ln(\text{OR})$: $SE = \sqrt{\frac{1}{a} + \frac{1}{b} + \frac{1}{c} + \frac{1}{d}}$, with $95\%$ Wald CI: $\exp(\ln(\text{OR}) \pm 1.95996 \cdot SE)$.
- **Nature of Estimate:** These are **crude (unadjusted bivariate)** odds ratios.


In [ ]:
def categorize_trust(val):
    if pd.isna(val):
        return np.nan
    v = str(val)
    if v in ["Highly trust", "Somewhat trust"]:
        return "Trust"
    elif v in ["Somewhat distrust", "Highly distrust"]:
        return "Distrust"
    elif v == "Neither trust nor distrust":
        return "Neutral"
    return np.nan

df["AIAcc_tier"] = df["AIAcc"].apply(categorize_trust)

def categorize_exp(val):
    if pd.isna(val):
        return np.nan
    try:
        x = float(val)
    except (ValueError, TypeError):
        return np.nan
    if x < 2.0:
        return "1. <2 years (Entry)"
    elif x < 6.0:
        return "2. 2 to 5 years (Junior)"
    elif x < 11.0:
        return "3. 6 to 10 years (Mid)"
    elif x < 21.0:
        return "4. 11 to 20 years (Senior)"
    else:
        return "5. 21+ years (Veteran)"

df["WorkExp_bracket"] = df["WorkExp_num"].apply(categorize_exp)

df_exp = df[df["AIAcc_tier"].notna() & df["WorkExp_bracket"].notna()].copy()

# 1. Global 5x3 Contingency Table
ct_5x3 = pd.crosstab(df_exp["WorkExp_bracket"], df_exp["AIAcc_tier"])
obs_5x3 = ct_5x3.values
row_sums_5x3 = obs_5x3.sum(axis=1, keepdims=True)
col_sums_5x3 = obs_5x3.sum(axis=0, keepdims=True)
n_total_5x3 = obs_5x3.sum()
exp_5x3 = (row_sums_5x3 @ col_sums_5x3) / n_total_5x3
chi2_5x3 = float(np.sum((obs_5x3 - exp_5x3) ** 2 / exp_5x3))
df_5x3 = (obs_5x3.shape[0] - 1) * (obs_5x3.shape[1] - 1)
p_val_5x3 = chi2_p_value(chi2_5x3, df_5x3)
cramers_v_5x3 = float(np.sqrt(chi2_5x3 / (n_total_5x3 * min(obs_5x3.shape[0] - 1, obs_5x3.shape[1] - 1))))
exp_5x3_df = pd.DataFrame(exp_5x3, index=ct_5x3.index, columns=ct_5x3.columns)

print("--- 1. Global 5x3 Contingency Table (Observed Frequencies) ---")
print(ct_5x3)
print(f"\nGlobal Chi-Square Test: chi2 = {chi2_5x3:.4f}, df = {df_5x3}, p = {p_val_5x3:.6e}, Cramer's V = {cramers_v_5x3:.5f}")
print("\n--- Global 5x3 Expected Frequencies ---")
print(exp_5x3_df.round(2))

# 2. Collapsed 5x2 Contingency Table (Distrust vs Not Distrust)
df_exp["Distrust_binary"] = df_exp["AIAcc_tier"].apply(
    lambda x: "Distrust" if x == "Distrust" else "Not Distrust"
)
ct_5x2 = pd.crosstab(df_exp["WorkExp_bracket"], df_exp["Distrust_binary"])
obs_5x2 = ct_5x2.values
row_sums_5x2 = obs_5x2.sum(axis=1, keepdims=True)
col_sums_5x2 = obs_5x2.sum(axis=0, keepdims=True)
exp_5x2 = (row_sums_5x2 @ col_sums_5x2) / n_total_5x3
chi2_5x2 = float(np.sum((obs_5x2 - exp_5x2) ** 2 / exp_5x2))
df_5x2 = (obs_5x2.shape[0] - 1) * (obs_5x2.shape[1] - 1)
p_val_5x2 = chi2_p_value(chi2_5x2, df_5x2)
cramers_v_5x2 = float(np.sqrt(chi2_5x2 / (n_total_5x3 * min(obs_5x2.shape[0] - 1, obs_5x2.shape[1] - 1))))
exp_5x2_df = pd.DataFrame(exp_5x2, index=ct_5x2.index, columns=ct_5x2.columns)

print("\n--- 2. Collapsed 5x2 Contingency Table (Distrust vs Not Distrust) ---")
print(ct_5x2)
print(f"\nCollapsed Chi-Square Test: chi2 = {chi2_5x2:.4f}, df = {df_5x2}, p = {p_val_5x2:.6e}, Cramer's V = {cramers_v_5x2:.5f}")
print("\n--- Collapsed 5x2 Expected Frequencies ---")
print(exp_5x2_df.round(2))


In [ ]:
# Calculate Audited Crude Odds Ratios (Reference: Entry <2 yrs)
ref_group = "1. <2 years (Entry)"
ref_d = int(ct_5x2.loc[ref_group, "Distrust"])
ref_nd = int(ct_5x2.loc[ref_group, "Not Distrust"])
ref_odds = ref_d / ref_nd

seniority_rows = []
for bracket in sorted(df_exp["WorkExp_bracket"].unique()):
    d = int(ct_5x2.loc[bracket, "Distrust"])
    nd = int(ct_5x2.loc[bracket, "Not Distrust"])
    tot = d + nd
    pct = d / tot * 100.0
    odds = d / nd
    or_val = odds / ref_odds
    se = math.sqrt(1.0 / d + 1.0 / nd + 1.0 / ref_d + 1.0 / ref_nd) if bracket != ref_group else 0.0
    ci_l = math.exp(math.log(or_val) - 1.959964 * se) if bracket != ref_group else 1.0
    ci_u = math.exp(math.log(or_val) + 1.959964 * se) if bracket != ref_group else 1.0
    z = (math.log(or_val) / se) if (bracket != ref_group and se > 0) else 0.0
    p = normal_p_value(z) if bracket != ref_group else 1.0

    seniority_rows.append({
        "Bracket": bracket, "Distrust (a)": d, "Not Distrust (b)": nd,
        "Total (n)": tot, "% Distrust": round(pct, 2),
        "Crude OR": round(or_val, 3), "95% CI": f"[{ci_l:.3f}, {ci_u:.3f}]",
        "z-score": round(z, 2), "p-value": p,
    })

df_seniority = pd.DataFrame(seniority_rows)
print("\nAudited Crude Seniority Odds Ratios (Distrust vs Not Distrust):")
print(df_seniority.to_string(index=False))


### 2.2 Language Associations with Multiple Hypothesis Testing Correction
We evaluate whether stating active use of specific programming languages in `LanguageHaveWorkedWith` is associated with higher reported distrust. 
To control the family-wise error rate and false discovery rate across the family of $m = 6$ language tests, we report both **Benjamini–Hochberg (BH) FDR adjusted p-values** and **Bonferroni adjusted p-values**.


In [ ]:
df_lang = df[df["AIAcc_tier"].notna() & df["LanguageHaveWorkedWith"].notna()].copy()
n_lang = len(df_lang)

key_languages = ["Rust", "C++", "Go", "JavaScript", "TypeScript", "Python"]
lang_rows = []

for lang in key_languages:
    is_u = df_lang["LanguageHaveWorkedWith"].apply(lambda x: 1 if lang in [t.strip() for t in str(x).split(";")] else 0)
    a = int(((is_u == 1) & (df_lang["AIAcc_tier"] == "Distrust")).sum())
    b = int(((is_u == 1) & (df_lang["AIAcc_tier"] != "Distrust")).sum())
    c = int(((is_u == 0) & (df_lang["AIAcc_tier"] == "Distrust")).sum())
    d = int(((is_u == 0) & (df_lang["AIAcc_tier"] != "Distrust")).sum())
    u_tot = a + b
    nu_tot = c + d

    or_val = (a * d) / (b * c)
    se = math.sqrt(1.0 / a + 1.0 / b + 1.0 / c + 1.0 / d)
    ci_l = math.exp(math.log(or_val) - 1.959964 * se)
    ci_u = math.exp(math.log(or_val) + 1.959964 * se)
    z = math.log(or_val) / se

    obs = np.array([[a, b], [c, d]], dtype=float)
    exp = np.outer(obs.sum(axis=1), obs.sum(axis=0)) / n_lang
    chi2_val = float(np.sum((obs - exp) ** 2 / exp))
    raw_p = chi2_p_value(chi2_val, 1)

    lang_rows.append({
        "Language": lang, "Users (n)": u_tot, "User % Distrust": round(a / u_tot * 100, 2),
        "Non-Users (n)": nu_tot, "Non-User % Distrust": round(c / nu_tot * 100, 2),
        "Crude OR": round(or_val, 3), "95% CI": f"[{ci_l:.3f}, {ci_u:.3f}]",
        "Chi2": round(chi2_val, 2), "Raw p-value": raw_p,
    })

raw_p_list = [r["Raw p-value"] for r in lang_rows]
bh_p_list = adjust_p_values(raw_p_list, method="fdr_bh")
bonf_p_list = adjust_p_values(raw_p_list, method="bonferroni")

for idx, r in enumerate(lang_rows):
    r["BH FDR p-value"] = bh_p_list[idx]
    r["Bonferroni p-value"] = bonf_p_list[idx]

df_lang_table = pd.DataFrame(lang_rows).sort_values(by="Raw p-value")
print(f"Language Associations with AIAcc Distrust (Complete Cases N = {n_lang:,}):")
print(df_lang_table[["Language", "Users (n)", "User % Distrust", "Non-Users (n)", "Non-User % Distrust", "Crude OR", "95% CI", "Chi2", "Raw p-value", "BH FDR p-value", "Bonferroni p-value"]].to_string(index=False))


### 2.3 Substantive Interpretation (Descriptive Observational Language)
- **Seniority:** Veteran developers (21+ years) exhibit higher odds of distrust in AI output accuracy relative to entry-level peers ($OR = 1.178$ $[1.053, 1.318]$, $p = 0.0043$).
- **Language Stacks:** Stated use of strict systems programming languages is strongly and positively associated with reported distrust in AI accuracy:
  - **Rust users:** $63.63\%$ distrust vs. $44.34\%$ among non-users ($OR = 2.196$ $[2.059, 2.343]$, BH FDR $p = 1.48 \times 10^{-128}$).
  - **C++ users:** $54.65\%$ distrust vs. $44.95\%$ among non-users ($OR = 1.476$ $[1.400, 1.556]$, BH FDR $p = 6.84 \times 10^{-47}$).
  - **Go users:** $52.94\%$ distrust vs. $46.11\%$ among non-users ($OR = 1.314$ $[1.237, 1.396]$, BH FDR $p = 1.17 \times 10^{-18}$).
  - In contrast, developers working in web ecosystems (JavaScript, TypeScript) exhibit lower crude odds of distrust relative to non-users ($OR = 0.851$ and $OR = 0.908$, respectively).
- **Non-Causal Note:** These findings represent cross-sectional observational associations. Programming languages do not causally determine psychological trust; rather, developers working on complex systems tasks involving memory safety, concurrency, and rigid type systems operate in domains where hallucinated code is immediately rejected by compilers.


## Section 3: Research Question 2 (RQ2) — Compensation and Technology Associations

### 3.1 Econometric Specification & Methodological Clarification
We estimate conditional compensation associations using a multivariable Mincerian wage model on natural log annual compensation:

$$\ln(\text{Salary}_i) = \beta_0 + \sum_{k=1}^6 \beta_k \text{Tech}_{ki} + \gamma_1 \text{WorkExp}_i + \gamma_2 \text{WorkExp}_i^2 + \sum_{j=1}^7 \delta_j \text{Country}_{ji} + \sum_{m=1}^5 \lambda_m \text{DevType}_{mi} + \varepsilon_i$$

- **Estimation Sample:** $N = 22,769$ qualified respondents with complete covariate data, restricted to annual compensation between $\$1,000$ and $\$1,000,000$ USD to eliminate nominal data-entry errors and unrepresentative outliers.
- **Design Matrix Dimensions:** $k = 20$ explanatory variables (6 technology indicators, 2 experience terms, 7 country indicators, 5 developer role indicators: Full-stack developer, Back-end developer, Front-end developer, Engineering manager, DevOps engineer or professional) and $p = 21$ estimated parameters including Intercept.
- **Degrees of Freedom:** $df_{\text{model}} = 20$, $df_{\text{residual}} = n - p = 22,769 - 21 = 22,748$.
- **Conversion to Percentage:** Coefficients are converted to percentage differences using the exact formula:
  $$\%\Delta = 100 \times (e^{\hat{\beta}} - 1)$$
- **Observational Status:** All coefficients represent **conditional compensation associations among surveyed respondents, controlling for the specified covariates**. They are **not causal technology premiums or genuine market returns**.


In [ ]:
comp_min = 1000.0
comp_max = 1000000.0
df_m = df[
    df["ConvertedCompYearly"].notna()
    & (df["ConvertedCompYearly"] >= comp_min)
    & (df["ConvertedCompYearly"] <= comp_max)
    & df["WorkExp_num"].notna()
].copy()

df_m["log_salary"] = np.log(df_m["ConvertedCompYearly"])
df_m["WorkExp_num_sq"] = df_m["WorkExp_num"] ** 2

tech_vars = ["Tech_Rust", "Tech_Go", "Tech_Python", "Tech_TypeScript", "Tech_AWS", "Tech_Docker"]
df_m["Tech_Rust"] = df_m["LanguageHaveWorkedWith"].apply(lambda x: 1.0 if pd.notna(x) and "Rust" in [t.strip() for t in str(x).split(";")] else 0.0)
df_m["Tech_Go"] = df_m["LanguageHaveWorkedWith"].apply(lambda x: 1.0 if pd.notna(x) and "Go" in [t.strip() for t in str(x).split(";")] else 0.0)
df_m["Tech_Python"] = df_m["LanguageHaveWorkedWith"].apply(lambda x: 1.0 if pd.notna(x) and "Python" in [t.strip() for t in str(x).split(";")] else 0.0)
df_m["Tech_TypeScript"] = df_m["LanguageHaveWorkedWith"].apply(lambda x: 1.0 if pd.notna(x) and "TypeScript" in [t.strip() for t in str(x).split(";")] else 0.0)
df_m["Tech_AWS"] = df_m["PlatformHaveWorkedWith"].apply(lambda x: 1.0 if pd.notna(x) and "Amazon Web Services (AWS)" in str(x) else 0.0)
df_m["Tech_Docker"] = df_m["PlatformHaveWorkedWith"].apply(lambda x: 1.0 if pd.notna(x) and "Docker" in str(x) else 0.0)

top_countries = ["United States of America", "Germany", "United Kingdom of Great Britain and Northern Ireland", "Canada", "India", "France", "Netherlands"]
df_m["Country_grouped"] = df_m["Country"].apply(lambda c: c if c in top_countries else "Other")

top_devtypes = ["Developer, full-stack", "Developer, back-end", "Developer, front-end", "Engineering manager", "DevOps engineer or professional"]
df_m["DevType_grouped"] = df_m["DevType"].apply(lambda d: d if d in top_devtypes else "Other")

m3_cols = ["log_salary"] + tech_vars + ["WorkExp_num", "WorkExp_num_sq", "Country_grouped", "DevType_grouped"]
df_m3 = df_m[m3_cols].dropna().copy()
n_m3 = len(df_m3)

m3 = fit_ols(
    df_m3,
    y_col="log_salary",
    x_num_cols=tech_vars + ["WorkExp_num", "WorkExp_num_sq"],
    x_cat_cols=["Country_grouped", "DevType_grouped"],
    reference_categories={"Country_grouped": "Other", "DevType_grouped": "Other"},
)

print(f"Mincerian OLS Complete Cases: N = {n_m3:,}")
print(f"Goodness of Fit: R^2 = {m3['r_squared']:.4f}, Adjusted R^2 = {m3['adj_r_squared']:.4f}, Conventional OLS F({m3['df_model']}, {m3['df_residual']}) = {m3['f_statistic']:.2f}")
print(f"Breusch-Pagan Test: LM = {m3['breusch_pagan_stat']:.2f}, df = {m3['breusch_pagan_df']}, p = {m3['breusch_pagan_p_value']:.4e} -> Heteroskedasticity detected; HC1 robust SEs reported.")


In [ ]:
# Evaluate Technology Indicators with Conventional vs HC1 Robust Standard Errors and Multiple Testing Adjustment
tech_eval_rows = []
for t in tech_vars:
    r = m3["coefficients"][m3["coefficients"]["term"] == t].iloc[0]
    tech_eval_rows.append({
        "Technology": t.replace("Tech_", ""),
        "beta": round(float(r["coef"]), 4),
        "Controlled % Diff": round(float(r["pct_effect"]), 2),
        "Conv SE": round(float(r["std_error"]), 4),
        "Robust HC1 SE": round(float(r["robust_se"]), 4),
        "Robust t-stat": round(float(r["robust_t_stat"]), 2),
        "Robust 95% CI": f"[{r['pct_robust_ci_lower']:+.2f}%, {r['pct_robust_ci_upper']:+.2f}%]",
        "Raw Robust p-value": float(r["robust_p_value"]),
    })

raw_robust_p = [r["Raw Robust p-value"] for r in tech_eval_rows]
bh_robust_p = adjust_p_values(raw_robust_p, method="fdr_bh")
bonf_robust_p = adjust_p_values(raw_robust_p, method="bonferroni")

for idx, r in enumerate(tech_eval_rows):
    r["BH FDR p-val"] = bh_robust_p[idx]
    r["Bonferroni p-val"] = bonf_robust_p[idx]

df_tech_summary = pd.DataFrame(tech_eval_rows).sort_values(by="Raw Robust p-value")
print("Conditional Technology Compensation Associations (Controlled for Exp, Country, Role):")
print(df_tech_summary[["Technology", "Controlled % Diff", "Conv SE", "Robust HC1 SE", "Robust t-stat", "Robust 95% CI", "Raw Robust p-value", "BH FDR p-val", "Bonferroni p-val"]].to_string(index=False))


In [ ]:
# Cook's Distance and Leverage Diagnostics
p_diag = len(m3["design_matrix_cols"])
n_diag = m3["n_obs"]
s2_diag = float(m3["residual_std_err"]) ** 2
e_diag = m3["residuals"]

# Construct X explicitly for leverage
X_diag_list = [np.ones((n_diag, 1))]
for t in tech_vars + ["WorkExp_num", "WorkExp_num_sq"]:
    X_diag_list.append(df_m3[t].values.reshape(-1, 1))

for c in sorted(df_m3["Country_grouped"].unique()):
    if c != "Other":
        X_diag_list.append((df_m3["Country_grouped"] == c).astype(float).values.reshape(-1, 1))

for d in sorted(df_m3["DevType_grouped"].unique()):
    if d != "Other":
        X_diag_list.append((df_m3["DevType_grouped"] == d).astype(float).values.reshape(-1, 1))

X_diag = np.hstack(X_diag_list)
XtX_inv_diag = np.linalg.pinv(X_diag.T @ X_diag)
H_diag = np.sum((X_diag @ XtX_inv_diag) * X_diag, axis=1)

# Cook's distance
cooks_d = (e_diag**2 / (p_diag * s2_diag)) * (H_diag / (1.0 - H_diag)**2)
cd_thresh = 4.0 / n_diag
n_exceed_cd = int(np.sum(cooks_d > cd_thresh))

h_thresh = 2.0 * p_diag / n_diag
n_exceed_h = int(np.sum(H_diag > h_thresh))

print(f"Leverage Threshold 2p/n = {h_thresh:.6f}: Exceeding = {n_exceed_h:,} ({n_exceed_h/n_diag*100:.2f}%)")
print(f"Cook's D Threshold 4/n = {cd_thresh:.6e}: Exceeding = {n_exceed_cd:,} ({n_exceed_cd/n_diag*100:.2f}%)")
print(f"Cook's D Distribution: Min={np.min(cooks_d):.2e}, 25%={np.percentile(cooks_d, 25):.2e}, Median={np.median(cooks_d):.2e}, 75%={np.percentile(cooks_d, 75):.2e}, Max={np.max(cooks_d):.4f}")

# Sensitivity Analysis: Refit Model 3 excluding influential observations (D_i > 4/n)
df_m3["cooks_d"] = cooks_d
df_m3_clean = df_m3[df_m3["cooks_d"] <= cd_thresh].copy()
m3_clean = fit_ols(
    df_m3_clean,
    y_col="log_salary",
    x_num_cols=tech_vars + ["WorkExp_num", "WorkExp_num_sq"],
    x_cat_cols=["Country_grouped", "DevType_grouped"],
    reference_categories={"Country_grouped": "Other", "DevType_grouped": "Other"},
)

sens_rows = []
for t in tech_vars:
    row_f = m3["coefficients"][m3["coefficients"]["term"] == t].iloc[0]
    row_c = m3_clean["coefficients"][m3_clean["coefficients"]["term"] == t].iloc[0]
    b_f = float(row_f["coef"])
    b_c = float(row_c["coef"])
    diff = (b_c - b_f) / b_f * 100.0
    sens_rows.append({
        "Technology": t.replace("Tech_", ""),
        "Full Beta": round(b_f, 4),
        "Full Robust SE": round(float(row_f["robust_se"]), 4),
        "Robust Beta (D_i <= 4/n)": round(b_c, 4),
        "Clean Robust SE": round(float(row_c["robust_se"]), 4),
        "Beta Diff %": round(diff, 2),
    })

print(f"\nSensitivity Model N = {len(df_m3_clean):,}, R^2 = {m3_clean['r_squared']:.4f}, Clean F = {m3_clean['f_statistic']:.1f}")
print(pd.DataFrame(sens_rows).to_string(index=False))


### 3.2 Model Diagnostics & Limitations
- **Salary Retention Rate:** Among qualified respondents with positive annual compensation ($N = 23,947$), trimming the extreme tails at $\$1,000$ and $\$1,000,000$ USD removes $724$ low records and $46$ high records (total $770$ trimmed, $3.22\%$), retaining $N = 23,177$ records ($96.78\% \approx 96.8\%$). After dropping incomplete covariates, the final estimation sample contains $N = 22,769$ complete cases.
- **Test Statistic Clarification:** The overall model test statistic reported ($F(20, 22748) = 710.77$) is the **conventional OLS $F$-statistic**, not a robust Wald test.
- **Heteroskedasticity:** The Breusch-Pagan test confirms significant heteroskedasticity ($LM = 2,269.63, df = 20, p < 10^{-300}$), validating the deployment of White HC1 robust standard errors.
- **Multicollinearity:** All technology VIF values range between $1.08$ and $1.34$, verifying low multicollinearity risk.
- **Leverage Diagnostics:** Using threshold $h_{ii} > 2p/n = 2(21)/22,769 = 0.001845$, exactly $1,797$ observations ($7.89\%$) exhibit elevated leverage.
- **Influence & Sensitivity Analysis:** Observations with Cook's distance exceeding the diagnostic screening threshold $4/n = 4/22,769 \approx 1.76 \times 10^{-4}$ ($1,039$ observations, $4.56\%$) are not invalid records, but exceed a screening cutoff. Re-estimating Model 3 on the subset within the threshold ($N = 21,730$, $R^2 = 0.4845$, Clean $F = 1,020.2$) serves as an exploratory sensitivity check, not a definitive or inherently cleaner model. Across all six evaluated technologies, coefficient signs and significance classifications are preserved, but coefficient magnitudes and relative rankings change noticeably. Specifically, Docker's coefficient decreases by $23.18\%$ (from $+0.1237$, rank 1 in the full model, to $+0.0950$, rank 2 in the screened refit), while Go's coefficient increases by $37.22\%$ (from $+0.0686$, rank 5 in the full model, to $+0.0942$, rank 3 in the screened refit), making Rust the highest-ranked technology premium ($+0.1217$, $+1.17\%$). This sensitivity refit demonstrates that point estimates and rankings are sensitive to observations exceeding the diagnostic screening threshold, and does not establish that estimates are unaffected by influential observations.
- **Omitted Variable Limitations:** Unmeasured factors (e.g., individual programming aptitude, firm profitability tier, equity grants, and local cost-of-living differences) remain in the error term.


## Section 4: Research Question 3 (RQ3) — AI Agent Adoption and Workplace Outcomes

### 4.1 Job Satisfaction ANOVA & Effect Size Accounting
We analyze developer role satisfaction (`JobSat`, 0–10 integer scale) across the 6-tier adoption hierarchy among $N = 23,766$ complete cases.
- **Effect Sizes:**
  - Eta-squared: $\eta^2 = \frac{SS_{\text{between}}}{SS_{\text{total}}}$
  - Omega-squared: $\omega^2 = \frac{SS_{\text{between}} - df_{\text{between}} \cdot MS_{\text{within}}}{SS_{\text{total}} + MS_{\text{within}}}$
- **Post-Hoc Pairwise Comparisons:** Evaluated against Non-Planners ($N = 8,711$) using pooled standard error and Bonferroni adjustment.


In [ ]:
agent_tiers = [
    "Yes, I use AI agents at work daily",
    "Yes, I use AI agents at work weekly",
    "Yes, I use AI agents at work monthly or infrequently",
    "No, I use AI exclusively in copilot/autocomplete mode",
    "No, but I plan to",
    "No, and I don't plan to",
]

df_agent = df[df["AIAgents"].isin(agent_tiers)].copy()
df_jobsat = df_agent[df_agent["JobSat"].notna()].copy()
df_jobsat["JobSat"] = df_jobsat["JobSat"].astype(float)
res_js = compare_groups_continuous(df_jobsat, "AIAgents", "JobSat")

print(f"One-Way ANOVA: JobSat across AIAgent Tiers (N = {res_js['total_n']:,}):")
print(f"  SS_between = {res_js['ss_between']:.2f}, df_between = {res_js['df_between']}")
print(f"  SS_within  = {res_js['ss_within']:.2f}, df_within  = {res_js['df_within']}")
print(f"  F-statistic = {res_js['anova_f_stat']:.4f}, p-value = {res_js['anova_p_value']:.4e}")
print(f"  Eta-squared (eta^2)     = {res_js['eta_squared']:.6f} ({res_js['eta_squared']*100:.3f}% of variance explained)")
print(f"  Omega-squared (omega^2) = {res_js['omega_squared']:.6f} ({res_js['omega_squared']*100:.3f}% of variance explained)")


In [ ]:
# Post-Hoc Pairwise Comparisons vs Non-Planners
nonplan_vals = df_jobsat[df_jobsat["AIAgents"] == "No, and I don't plan to"]["JobSat"].values
m_ref = float(np.mean(nonplan_vals))
ms_w = res_js["ss_within"] / res_js["df_within"]
n_ref = len(nonplan_vals)

posthoc_rows = []
for g in agent_tiers:
    if g == "No, and I don't plan to":
        continue
    g_vals = df_jobsat[df_jobsat["AIAgents"] == g]["JobSat"].values
    m_g = float(np.mean(g_vals))
    n_g = len(g_vals)
    diff = m_g - m_ref
    se_d = math.sqrt(ms_w * (1.0 / n_g + 1.0 / n_ref))
    t_d = diff / se_d
    p_d = normal_p_value(t_d)
    d_cohen = diff / math.sqrt(ms_w)

    posthoc_rows.append({
        "Adoption Tier": g, "N": n_g, "Mean": round(m_g, 3),
        "Diff vs Non-Plan": round(diff, 3), "SE": round(se_d, 4),
        "95% CI": f"[{diff - 1.959964 * se_d:+.3f}, {diff + 1.959964 * se_d:+.3f}]",
        "Cohen's d": round(d_cohen, 3), "Raw p-value": p_d,
        "Bonferroni p-val": min(1.0, p_d * 5),
    })

df_posthoc_table = pd.DataFrame(posthoc_rows).sort_values(by="Raw p-value")
print("Post-Hoc Pairwise Comparisons vs Non-Planners Baseline (Mean = 7.149):")
print(df_posthoc_table.to_string(index=False))


### 4.2 Practical Magnitude Interpretation
While the ANOVA F-test is highly statistically significant ($F = 11.62, p = 3.02 \times 10^{-11}$), the effect size metrics reveal that:
- **$\eta^2 = 0.0024$** and **$\omega^2 = 0.0022$**: Autonomous AI agent adoption accounts for less than **$0.25\%$** of the total variance in developer job satisfaction.
- The difference of **$+0.252$ points** on a 10-point scale corresponds to a Cohen's $d$ of **$0.128$** (roughly one-eighth of a standard deviation). In practical terms, this is a **small effect size** that should not be overstated.

### 4.3 Audit of the Favorable Sentiment Odds Ratio ($OR = 21.29$)
We audit the reported Odds Ratio of $21.29$ for favorable AI sentiment (`AISent`) comparing Daily Agent Users vs. Non-Planners:
- **Source Fields:** `AISent` and `AIAgents`.
- **Classification Rule:** Favorable = "Very favorable" or "Favorable"; Non-Favorable = "Indifferent", "Unfavorable", "Very unfavorable", "Unsure".


In [ ]:
df_sent = df_agent[df_agent["AISent"].notna()].copy()
df_sent["AISent_fav"] = df_sent["AISent"].isin(["Very favorable", "Favorable"]).astype(int)

daily_m = df_sent["AIAgents"] == "Yes, I use AI agents at work daily"
nonplan_m = df_sent["AIAgents"] == "No, and I don't plan to"

a_s = int(((daily_m) & (df_sent["AISent_fav"] == 1)).sum())
b_s = int(((daily_m) & (df_sent["AISent_fav"] == 0)).sum())
c_s = int(((nonplan_m) & (df_sent["AISent_fav"] == 1)).sum())
d_s = int(((nonplan_m) & (df_sent["AISent_fav"] == 0)).sum())

or_s = (a_s * d_s) / (b_s * c_s)
se_s = math.sqrt(1.0 / a_s + 1.0 / b_s + 1.0 / c_s + 1.0 / d_s)
ci_l_s = math.exp(math.log(or_s) - 1.959964 * se_s)
ci_u_s = math.exp(math.log(or_s) + 1.959964 * se_s)

print("2x2 Contingency Table (Daily Agent Users vs Non-Planners):")
print(f"                      Favorable   Non-Favorable   Total      % Favorable")
print(f"  Daily Agent Users:  a = {a_s:,}     b = {b_s:,}        {a_s+b_s:,}     {a_s/(a_s+b_s)*100:.2f}%")
print(f"  Non-Planners:       c = {c_s:,}     d = {d_s:,}       {c_s+d_s:,}     {c_s/(c_s+d_s)*100:.2f}%")
print(f"\nCalculated Odds Ratio = ({a_s} * {d_s}) / ({b_s} * {c_s}) = {or_s:.3f}")
print(f"95% Wald CI: [{ci_l_s:.3f}, {ci_u_s:.3f}], SE(ln OR) = {se_s:.4f}")
print("Separation Check: All 4 cells contain > 400 observations. Zero sparse-cell bias or separation.")


## Section 5: Research Question 4 (RQ4) — Remote Work Dynamics, Selection Missingness, and Compensation

### 5.1 Missingness Terminology & Contingency Audit
Questionnaire routing audit confirms that the public survey schema (`survey_results_schema.csv`) releases question texts and identifiers (`QID24` for `RemoteWork`, `QID3` for `MainBranch`), but does not publish the underlying branching script. Therefore, the observed association between `MainBranch` and `RemoteWork` nonresponse is an **empirically observed pattern in survey response data**, rather than a published branching specification.

> **Mandated Methodological Finding:**  
> RemoteWork nonresponse is associated with respondent branch. The observed association suggests that missingness is not MCAR with respect to MainBranch. The available data alone cannot distinguish MAR from MNAR.


In [ ]:
df["RemoteWork_status"] = df["RemoteWork"].isna().map({True: "Missing", False: "Answered"})
ct_branch = pd.crosstab(df["MainBranch"], df["RemoteWork_status"], margins=True)
pct_branch = pd.crosstab(df["MainBranch"], df["RemoteWork_status"], normalize="index") * 100

print("Contingency Table: MainBranch by RemoteWork Status (Counts):")
print(ct_branch)
print("\nMissingness Rates within MainBranch (%):")
print(pct_branch.round(2))


### 5.2 Sensitivity Analysis for Complete Cases
To determine whether complete-case analysis ($N = 20,236$) introduces selection bias relative to the broader population of salary earners, we compare salary earners who answered `RemoteWork` vs. those who skipped `RemoteWork`.


In [ ]:
df_sal = df[
    df["ConvertedCompYearly"].notna()
    & (df["ConvertedCompYearly"] >= comp_min)
    & (df["ConvertedCompYearly"] <= comp_max)
].copy()

df_sal["has_rw"] = df_sal["RemoteWork"].notna()
sal_rw = df_sal[df_sal["has_rw"]]
sal_norw = df_sal[~df_sal["has_rw"]]

print(f"Salary Earners ($1k-$1M USD): Total N = {len(df_sal):,}")
print(f"  Answered RemoteWork: N = {len(sal_rw):,} ({len(sal_rw)/len(df_sal)*100:.2f}%)")
print(f"  Missing RemoteWork:   N = {len(sal_norw):,} ({len(sal_norw)/len(df_sal)*100:.2f}%)")
print("\nSensitivity Comparison:")
print(f"  Median Annual Salary: Answered = ${sal_rw['ConvertedCompYearly'].median():,.0f} vs Missing = ${sal_norw['ConvertedCompYearly'].median():,.0f}")
print(f"  Mean Experience (years): Answered = {sal_rw['WorkExp_num'].mean():.2f} vs Missing = {sal_norw['WorkExp_num'].mean():.2f}")
print(f"  US Respondent Share: Answered = {(sal_rw['Country'] == 'United States of America').mean()*100:.2f}% vs Missing = {(sal_norw['Country'] == 'United States of America').mean()*100:.2f}%")
print("\nConclusion: Complete-case respondents have comparable median compensation but slightly lower experience and higher US representation; the 20,236 complete cases should not be treated as a random sample of all respondents.")


### 5.3 Conditional Compensation Differences Across Working Arrangements
We evaluate conditional compensation differences relative to the in-person reference category, controlling for cumulative experience, organization size (`OrgSize`), and geography.

- **Model C Design Matrix & Diagnostics:**
  - Complete cases: $N = 20,236$ salary earners.
  - Explanatory variables: $k = 18$ ($2$ experience terms, $4$ remote work indicators, $5$ organization size indicators, $7$ country indicators).
  - Estimated parameters: $p = k + 1 = 18 + 1 = 19$ (including Intercept).
  - Degrees of freedom: $df_{\text{model}} = 18$, $df_{\text{residual}} = 20,236 - 19 = 20,217$.
  - Goodness of fit: $R^2 = 0.4254$, Adjusted $R^2 = 0.4249$.
  - Conventional OLS $F$-statistic: $F(18, 20217) = 831.44$ ($p < 10^{-300}$, conventional OLS $F$, not robust Wald $F$).
  - Breusch-Pagan test: $LM = 2,054.64, df = 18, p < 10^{-300}$ (White HC1 robust standard errors reported).


In [ ]:
df_rw_model = df[
    df["RemoteWork"].notna()
    & df["ConvertedCompYearly"].notna()
    & (df["ConvertedCompYearly"] >= comp_min)
    & (df["ConvertedCompYearly"] <= comp_max)
    & df["WorkExp_num"].notna()
    & df["OrgSize"].notna()
].copy()

df_rw_model["log_salary"] = np.log(df_rw_model["ConvertedCompYearly"])
df_rw_model["WorkExp_num_sq"] = df_rw_model["WorkExp_num"] ** 2

def group_org_size(val):
    if pd.isna(val):
        return "Unknown"
    v = str(val)
    if "Just me" in v or "Less than 20" in v:
        return "1. Micro (<20)"
    elif "20 to 99" in v:
        return "2. Small (20-99)"
    elif "100 to 499" in v or "500 to 999" in v:
        return "3. Mid-size (100-999)"
    elif "1,000 to 4,999" in v:
        return "4. Large (1000-4999)"
    elif "5,000 to 9,999" in v or "10,000 or more" in v:
        return "5. Enterprise (5000+)"
    return "Unknown"

df_rw_model["OrgSize_grouped"] = df_rw_model["OrgSize"].apply(group_org_size)
df_rw_model["Country_grouped"] = df_rw_model["Country"].apply(lambda c: c if c in top_countries else "Other")

m_rw = fit_ols(
    df_rw_model,
    y_col="log_salary",
    x_num_cols=["WorkExp_num", "WorkExp_num_sq"],
    x_cat_cols=["RemoteWork", "OrgSize_grouped", "Country_grouped"],
    reference_categories={"RemoteWork": "In-person", "OrgSize_grouped": "1. Micro (<20)", "Country_grouped": "Other"},
)

rw_rows = []
for term in m_rw["coefficients"]["term"]:
    if "RemoteWork_" in term:
        r = m_rw["coefficients"][m_rw["coefficients"]["term"] == term].iloc[0]
        rw_rows.append({
            "Work Arrangement (vs In-person)": term.replace("RemoteWork_", ""),
            "beta": round(float(r["coef"]), 4),
            "Controlled % Diff": round(float(r["pct_effect"]), 2),
            "Robust HC1 SE": round(float(r["robust_se"]), 4),
            "Robust t-stat": round(float(r["robust_t_stat"]), 2),
            "Robust 95% CI": f"[{r['pct_robust_ci_lower']:+.2f}%, {r['pct_robust_ci_upper']:+.2f}%]",
            "Robust p-value": float(r["robust_p_value"]),
        })

df_rw_results = pd.DataFrame(rw_rows)
print(f"Model C Complete Cases N = {len(df_rw_model):,}, R^2 = {m_rw['r_squared']:.4f}, Adj R^2 = {m_rw['adj_r_squared']:.4f}, Conventional OLS F({m_rw['df_model']}, {m_rw['df_residual']}) = {m_rw['f_statistic']:.2f}")
print("\nConditional Compensation Differences Relative to In-person Reference Category:")
print(df_rw_results.to_string(index=False))


## Section 6: Methodological Synthesis & Observational Boundaries

### 6.1 Summary of Core Findings Across Research Questions

| Research Question | Primary Focus | Key Metric & Multiple Testing Status | Substantive Observational Conclusion |
| :--- | :--- | :--- | :--- |
| **RQ1: AI Trust** | Perception of AI accuracy (`AIAcc`) | Rust Distrust = $63.63\%$ ($OR = 2.196$, BH FDR $p < 10^{-120}$) | Experienced developers exhibit higher odds of distrust. Statically typed, memory-safe systems languages are strongly associated with higher skepticism. |
| **RQ2: Tech Stack** | Compensation associations of specialized tech | Rust Controlled = $+12.79\%$ (Robust $p < 10^{-12}$); Python = $+1.32\%$ (n.s.) | Unadjusted wage gaps attenuate substantially under Mincerian controls. Rust and Go retain positive conditional associations; Python shows no significant conditional difference. |
| **RQ3: AI Agents** | Agent adoption vs sentiment & JobSat | Daily Agent JobSat = $7.401$ vs Non-Planners = $7.149$ (Bonferroni $p < 10^{-9}$) | Statistically significant elevation in job satisfaction ($+0.252$ points), but practical magnitude is small ($\eta^2 = 0.0024$, Cohen's $d = 0.128$). Daily agent users report $90.7\%$ favorable sentiment ($OR = 21.29$). |
| **RQ4: Remote Work** | Working arrangement conditional differences | Remote = $+30.65\%$ vs In-Person (Robust $p < 10^{-49}$) | In-person developers exhibit a conditional compensation difference of $\approx 30\%$ relative to remote and hybrid colleagues. Missingness is not MCAR with respect to MainBranch. |

### 6.2 Essential Observational Limitations
All reported findings are subject to core observational constraints:
1. **Strictly Non-Causal Associations:** Cross-sectional survey data cannot demonstrate causality. Associations reflect selection, sorting, and unmeasured heterogeneity.
2. **Volunteer Opt-In Self-Selection:** The Stack Overflow community survey is not a probability-sampled census; respondents are self-selected technical enthusiasts.
3. **Self-Reported Compensation:** Converted compensation figures are unverified by third-party payroll or tax microdata.
4. **Survivor Bias:** The dataset exclusively captures developers currently active in the workforce as of May–June 2025.

---
**Author:** Akash Das  
**Project Milestone:** Phase 5 — Analysis (Complete, Audited & Verified)
